# Классификация обращений в службу поддержки 

По тексту англоязычного обращения нужно предсказать одну из 77 категорий. Сравним три подхода: логистическую регрессию с TF-IDF, логистическую регрессию с эмбеддингами Word2Vec и дообученный трансформер.

Главный вопрос проекта - **почему модели путают категории и какие ошибки можно уменьшить**. Для этого разберём конкретные обращения, сформулируем гипотезы и проверим их на данных.

## Данные

Берём исходную часть train из [BANKING77](https://huggingface.co/datasets/PolyAI/banking77). Официальную часть test оставим для итоговой оценки. У каждого обращения есть текст и числовая метка. Добавим название категории (intent), чтобы удобнее читать примеры.

In [1]:
from datasets import load_dataset
from IPython.display import display

DATASET_NAME = "PolyAI/banking77"
SEED = 42

banking77_train = load_dataset(DATASET_NAME, split="train")
intent_names = banking77_train.features["label"].names

train_df = banking77_train.to_pandas()
train_df["intent"] = train_df["label"].map(dict(enumerate(intent_names)))

print(f"Обращений в train: {len(train_df):,}")
print(f"Категорий: {train_df['intent'].nunique()}")
display(train_df[["text", "intent"]].sample(5, random_state=SEED))

/opt/homebrew/Caskroom/miniforge/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 3080/3080 [00:00<00:00, 809680.75 examples/s]

Обращений в train: 10,003
Категорий: 77


,text,intent
6883,Is it possible for me to change my PIN number?,change_pin
5836,I'm not sure why my card didn't work,declined_card_payment
8601,I don't think my top up worked,top_up_failed
2545,Can you explain why my payment was charged a fee?,card_payment_fee_charged
8697,How long does a transfer from a UK account tak...,balance_not_updated_after_bank_transfer


## Проверка данных

Проверим исходную часть `train` перед разбиением на обучающую и валидационную выборки. Для поиска пустых строк и повторов уберём пробелы по краям текста в копии таблицы. Исходные обращения остаются в `train_df`.

### Пропуски и повторы

Отдельно считаем повторяющиеся пары «текст и категория» и тексты, которым присвоено несколько категорий. Если один и тот же текст окажется и в обучении, и в валидации, оценка может оказаться завышенной.

In [ ]:
audit_df = train_df[["text", "intent"]].copy()
audit_df["text"] = audit_df["text"].fillna("").str.strip()

empty_text = audit_df["text"].eq("")
missing_intent = audit_df["intent"].isna()

print(f"Обращений без текста: {empty_text.sum()}")
print(f"Обращений без категории: {missing_intent.sum()}")

# Сравниваем метки только там, где есть и текст, и категория.
labeled_texts = audit_df.loc[~empty_text & ~missing_intent]

pair_counts = labeled_texts.groupby(["text", "intent"]).size()
repeated_pairs = pair_counts[pair_counts > 1]
print(f"Повторяющихся пар «текст и категория»: {len(repeated_pairs)}")
print(f"Лишних строк в этих парах: {(repeated_pairs - 1).sum()}")

if not repeated_pairs.empty:
    display(repeated_pairs.rename("n_rows").reset_index().head(10))

intents_per_text = labeled_texts.groupby("text")["intent"].nunique()
conflicting_texts = intents_per_text[intents_per_text > 1].index
print(f"Текстов с несколькими категориями: {len(conflicting_texts)}")

if len(conflicting_texts) > 0:
    conflict_examples = labeled_texts.loc[labeled_texts["text"].isin(conflicting_texts[:5])]
    display(conflict_examples.drop_duplicates().sort_values(["text", "intent"]))

### Размеры категорий

Посчитаем число размеченных обращений в каждой категории и посмотрим, сколько примеров есть у самых маленьких и самых больших классов.

In [ ]:
class_counts = train_df["intent"].value_counts().reindex(intent_names, fill_value=0)
class_counts = class_counts.sort_values()

print(f"Примеров на категорию: от {class_counts.min()} до {class_counts.max()}")
print("Пять самых маленьких категорий:")
display(class_counts.head(5).rename("n_examples").to_frame())
print("Пять самых больших категорий:")
display(class_counts.tail(5).sort_values(ascending=False).rename("n_examples").to_frame())

### Длина обращений

Оценим длину по числу частей текста, разделённых пробелами. Посмотрим на распределение и прочитаем несколько самых коротких обращений. Можно ли по ним понять намерение клиента?

In [ ]:
length_df = audit_df.loc[~empty_text].copy()
length_df["word_count"] = length_df["text"].str.split().str.len()

print("Распределение длины в словах:")
display(length_df["word_count"].describe().round(1).to_frame())
print("Медианная длина:", length_df["word_count"].median(), "слов")

print("Восемь самых коротких обращений:")
display(length_df.nsmallest(8, "word_count")[["text", "intent", "word_count"]])